In [ ]:
# ============================================================
# AMPER Intonation Visualizer and Batch Analysis
# ============================================================
#
# This script provides a compact workflow for:
#
# 1. Parsing AMPER .txt files containing syllable-based duration,
#    energy, and three F0 measurements per syllable.
# 2. Building raw AMPER F0 curves in absolute time.
# 3. Plotting declarative vs. interrogative contours in an
#    AMPER-like segment-based representation.
# 4. Scanning directory trees containing AMPER files stored in
#    folders named like "wav_tgrid_txt_*".
# 5. Running batch comparisons between affirmative ("a") and
#    interrogative ("i") sentence types.
# 6. Extracting two summary measures for statistical comparison:
#       - mean syllable duration in the exact first half of the utterance
#       - mean onset F0 over the first three syllables
#
# The script is designed for exploratory intonational analysis and for
# producing publication-ready comparative figures across locations,
# contexts, and repetitions.
#
# Requirements:
#   - numpy
#   - pandas
#   - matplotlib
#   - scipy
#
# Typical filename convention:
#   06b6bwka1.txt
#
# Interpreted as:
#   - locality: first 4 characters       -> 06b6
#   - context: middle substring          -> bwk
#   - sentence type: penultimate char    -> a / i
#   - repetition: final digit            -> 1
#
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from scipy import stats


# ============================================================
# 1. AMPER PARSING AND CURVE CONSTRUCTION
# ============================================================

def parse_amper_txt(path):
    """
    Parse an AMPER .txt file.

    Expected schematic structure:
        duration [ms]   energy [dB]   fo1 fo2 fo3 [Hz]
        1   70   71   186 186 188
        ...
        values at:
        2616 4169 5721 ...

    Returns
    -------
    dict
        A dictionary with:
        - 'syll'      : syllable indices (1..N)
        - 'dur_ms'    : syllable durations in ms
        - 'energy'    : syllable energy values in dB
        - 'f0_points' : array of shape (N, 3)
        - 'times_raw' : array of raw AMPER timestamps
    """
    syll_idx = []
    dur_ms = []
    energy = []
    f0_points = []
    times_raw = []

    mode = "start"

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue

            if "duration [ms]" in s and "fo3" in s:
                mode = "rows"
                continue

            if s.lower().startswith("values at"):
                mode = "times"
                continue

            if mode == "rows":
                parts = s.split()
                if not parts or not parts[0].isdigit():
                    continue
                if len(parts) < 5:
                    continue

                idx = int(parts[0])
                dur = float(parts[1])

                try:
                    en_val = float(parts[2])
                except ValueError:
                    en_val = np.nan

                f0_vals = []
                for x in parts[-3:]:
                    try:
                        f0_vals.append(float(x))
                    except ValueError:
                        f0_vals.append(np.nan)

                syll_idx.append(idx)
                dur_ms.append(dur)
                energy.append(en_val)
                f0_points.append(f0_vals)

            elif mode == "times":
                for tok in s.split():
                    if tok.isdigit():
                        times_raw.append(int(tok))

    syll = np.array(
        syll_idx if syll_idx else list(range(1, len(dur_ms) + 1)),
        dtype=int
    )
    dur_ms = np.array(dur_ms, dtype=float)
    energy = np.array(energy, dtype=float)
    f0_points = np.asarray(f0_points, dtype=float)
    times_raw = np.array(times_raw, dtype=float)

    return {
        "syll": syll,
        "dur_ms": dur_ms,
        "energy": energy,
        "f0_points": f0_points,
        "times_raw": times_raw,
    }


def build_amper_timebase(dur_ms, times_raw):
    """
    Build an absolute time axis in seconds from:
      - syllable durations in ms
      - raw AMPER timestamps

    Strategy
    --------
    1. Normalize times_raw to [0, 1].
    2. Rescale it to [0, total_duration_sec] so that the final
       duration matches the sum of syllable durations.
    """
    dur_ms = np.asarray(dur_ms, float)
    times_raw = np.asarray(times_raw, float)

    total_sec = float(dur_ms.sum()) / 1000.0 if dur_ms.size else 1.0

    if times_raw.size == 0:
        n = max(1, 3 * len(dur_ms))
        return np.linspace(0.0, total_sec, n)

    t_min = np.nanmin(times_raw)
    t_max = np.nanmax(times_raw)

    if not np.isfinite(t_min) or not np.isfinite(t_max) or t_max <= t_min:
        n = max(1, 3 * len(dur_ms))
        return np.linspace(0.0, total_sec, n)

    rel = (times_raw - t_min) / (t_max - t_min)
    return rel * total_sec


def build_amper_curve(path):
    """
    Convert an AMPER file into a raw F0 curve in absolute time.

    Returns
    -------
    dict
        - time_raw    : 3*N time values in seconds
        - f0_raw      : 3*N F0 values
        - speed_raw   : 3*N syllable-rate values (syll/s)
        - energy_raw  : 3*N energy values
        - rate_syll   : N syllable rates
        - syll_index  : syllable indices
        - dur_ms      : syllable durations in ms
        - energy_syll : syllable energy values
        - source      : source file path
    """
    data = parse_amper_txt(path)

    dur_ms = data["dur_ms"]
    energy_syll = data["energy"]
    f0_points = data["f0_points"]
    times_raw = data["times_raw"]

    n_syll = len(dur_ms)
    if n_syll == 0:
        raise ValueError(f"No syllables found in: {path}")

    time_raw = build_amper_timebase(dur_ms, times_raw)

    if time_raw.size != 3 * n_syll:
        print(
            f"Warning: expected {3 * n_syll} timestamps, found {time_raw.size} "
            f"in {os.path.basename(path)}. Rescaling using syllable count."
        )
        time_raw = np.linspace(time_raw.min(), time_raw.max(), 3 * n_syll)

    if f0_points.shape[0] != n_syll or f0_points.shape[1] != 3:
        raise ValueError(
            f"Unexpected F0 structure in {path}: expected (N, 3), "
            f"found {f0_points.shape}"
        )

    f0_raw = f0_points.reshape(-1)

    dur_sec = dur_ms / 1000.0
    with np.errstate(divide="ignore", invalid="ignore"):
        rate_syll = np.where(dur_sec > 0, 1.0 / dur_sec, 0.0)

    syll_for_point = np.repeat(np.arange(n_syll), 3)
    speed_raw = rate_syll[syll_for_point]
    energy_raw = energy_syll[syll_for_point]

    return {
        "time_raw": time_raw,
        "f0_raw": f0_raw,
        "speed_raw": speed_raw,
        "energy_raw": energy_raw,
        "rate_syll": rate_syll,
        "syll_index": data["syll"],
        "dur_ms": dur_ms,
        "energy_syll": energy_syll,
        "source": path,
    }


def smooth_amper_curve(curve, npoints=2000):
    """
    Interpolate a continuous F0 curve in absolute time using PCHIP.

    This is not the traditional AMPER display, but a shape-preserving
    continuous interpolation that avoids overshoot.

    Parameters
    ----------
    curve : dict
        Output of build_amper_curve().
    npoints : int
        Number of interpolation points.

    Returns
    -------
    dict
        Smoothed curve with time, f0, speed, and energy.
    """
    t = np.asarray(curve["time_raw"], float)
    f0 = np.asarray(curve["f0_raw"], float)
    sp = np.asarray(curve["speed_raw"], float)
    en = np.asarray(curve["energy_raw"], float)

    order = np.argsort(t)
    t = t[order]
    f0 = f0[order]
    sp = sp[order]
    en = en[order]

    uniq_t, uniq_idx = np.unique(t, return_index=True)
    f0u = f0[uniq_idx]
    spu = sp[uniq_idx]
    enu = en[uniq_idx]

    t_smooth = np.linspace(uniq_t.min(), uniq_t.max(), int(max(npoints, 200)))

    try:
        if uniq_t.size >= 3:
            spline_f0 = PchipInterpolator(uniq_t, f0u)
            f0_smooth = spline_f0(t_smooth)
        else:
            f0_smooth = np.interp(t_smooth, uniq_t, f0u)
    except Exception:
        t_smooth = uniq_t
        f0_smooth = f0u

    speed_smooth = np.interp(t_smooth, uniq_t, spu)
    energy_smooth = np.interp(t_smooth, uniq_t, enu)

    return {
        "time": t_smooth,
        "f0": f0_smooth,
        "speed": speed_smooth,
        "energy": energy_smooth,
        "source": curve.get("source", ""),
        "time_normalized": False,
    }


# ============================================================
# 2. SEGMENT-BASED PLOTTING
# ============================================================

def infer_sentence_label_from_name(path):
    """
    Infer a short contour label from the filename.

    Returns
    -------
    str
        'I' for interrogative-like names,
        'D' for declarative-like names,
        otherwise the first character of the stem.
    """
    base = os.path.basename(path)
    stem, _ = os.path.splitext(base)
    s = stem.lower()

    if "int" in s or "interr" in s or "_q" in s:
        return "I"
    if "dec" in s or "decl" in s or "_d" in s:
        return "D"

    return stem[:1].upper() if stem else "?"


def normalize_scalar(x, mn, mx):
    """
    Normalize a scalar to [0, 1].
    """
    if not np.isfinite(x) or not np.isfinite(mn) or not np.isfinite(mx) or mx <= mn:
        return 0.0
    return float((x - mn) / (mx - mn + 1e-12))


def plot_amper_segments_overlay(
    curves_raw,
    labels=None,
    output_path=None,
    f0_min=0,
    f0_max=300,
    title=None,
    centered=False
):
    """
    Plot AMPER-style segment-based contours using only syllable triplets.

    Parameters
    ----------
    curves_raw : list of dict
        Raw AMPER curves as returned by build_amper_curve().
    labels : list of str or None
        Display labels for the legend.
    output_path : str or None
        Optional path for saving the figure.
    f0_min, f0_max : float
        Plot y-axis limits.
    title : str or None
        Plot title.
    centered : bool
        If False, use absolute time on the x-axis.
        If True, align syllables by syllable position while preserving
        relative width based on syllable duration.
    """
    if not curves_raw:
        print("No curves available for plotting.")
        return

    if labels is None:
        visible_labels = []
        for i, c in enumerate(curves_raw):
            if i == 0:
                visible_labels.append("D")
            elif i == 1:
                visible_labels.append("I")
            else:
                visible_labels.append(infer_sentence_label_from_name(c.get("source", "")))
    else:
        visible_labels = list(labels)

    type_codes = [infer_sentence_label_from_name(c.get("source", "")) for c in curves_raw]

    if len(type_codes) >= 1 and type_codes[0] not in ("D", "I"):
        type_codes[0] = "D"
    if len(type_codes) >= 2 and type_codes[1] not in ("D", "I"):
        type_codes[1] = "I"

    all_speed = np.concatenate([np.asarray(c["speed_raw"], float) for c in curves_raw])
    all_energy = np.concatenate([np.asarray(c["energy_raw"], float) for c in curves_raw])
    speed_min, speed_max = np.nanmin(all_speed), np.nanmax(all_speed)
    energy_min, energy_max = np.nanmin(all_energy), np.nanmax(all_energy)

    all_f0 = np.concatenate([np.asarray(c["f0_raw"], float) for c in curves_raw])
    if f0_min is None:
        f0_min = float(np.nanmin(all_f0)) - 5.0
    if f0_max is None:
        f0_max = float(np.nanmax(all_f0)) + 5.0

    max_n_syll = 0
    for c in curves_raw:
        n_syll = min(len(c["dur_ms"]), len(c["f0_raw"]) // 3)
        max_n_syll = max(max_n_syll, n_syll)

    fig, ax = plt.subplots(figsize=(9, 4))

    for curve, visible_label, type_code in zip(curves_raw, visible_labels, type_codes):
        t = np.asarray(curve["time_raw"], float)
        f0 = np.asarray(curve["f0_raw"], float)
        sp = np.asarray(curve["speed_raw"], float)
        en = np.asarray(curve["energy_raw"], float)
        dur = np.asarray(curve["dur_ms"], float)

        if type_code == "D":
            cmap = plt.cm.Reds
        elif type_code == "I":
            cmap = plt.cm.Blues
        else:
            cmap = plt.cm.Purples

        n_syll = len(dur)
        max_syll_from_points = len(f0) // 3
        n_syll = min(n_syll, max_syll_from_points)

        if centered and n_syll > 0:
            mean_d = np.nanmean(dur)
            if not np.isfinite(mean_d) or mean_d <= 0:
                mean_d = 1.0
            dur_norm = dur / mean_d
            dur_norm = np.clip(dur_norm, 0.3, 1.7)
            base_width = 0.6
        else:
            dur_norm = None
            base_width = None

        for s_idx in range(n_syll):
            k0 = 3 * s_idx
            k1 = k0 + 1
            k2 = k0 + 2

            seg_f0 = f0[[k0, k1, k2]]

            if centered:
                center = s_idx + 1
                if dur_norm is not None and s_idx < dur_norm.size:
                    width = base_width * dur_norm[s_idx]
                else:
                    width = 0.6
                half_w = width / 2.0
                seg_x = np.array([center - half_w, center, center + half_w])
            else:
                seg_x = t[[k0, k1, k2]]

            sp_val = sp[k0]
            en_val = en[k0]

            s_norm = normalize_scalar(sp_val, speed_min, speed_max)
            e_norm = normalize_scalar(en_val, energy_min, energy_max)
            lw = 2.0 + 6.0 * e_norm
            color = cmap(0.2 + 0.8 * s_norm)

            ax.plot(
                seg_x,
                seg_f0,
                color=color,
                linewidth=lw,
                alpha=0.9,
                solid_capstyle="round",
                solid_joinstyle="round",
                antialiased=True,
            )

        mean_f0 = float(np.nanmean(f0))
        mean_dur = float(np.nanmean(curve["dur_ms"])) if len(curve["dur_ms"]) else np.nan

        legend_label = (
            f"{visible_label}\n"
            f"Mean f0: {mean_f0:.0f} Hz\n"
            f"Mean vowel dur: {mean_dur:.0f} ms"
        )

        ax.plot(
            [],
            [],
            color=cmap(0.8),
            linewidth=3.0,
            solid_capstyle="round",
            solid_joinstyle="round",
            label=legend_label
        )

    ax.set_ylim(f0_min, f0_max)

    if centered:
        ax.set_xlim(0.5, max_n_syll + 1.0)
        ax.set_xlabel("Syllable position")
    else:
        ax.set_xlabel("Time (s)")

    ax.set_ylabel("F0 (Hz)")

    if title:
        ax.set_title(title)

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        title="Contour",
        frameon=False,
    )

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches="tight")
        print(f"Saved plot: {os.path.basename(output_path)}")

    plt.show()


def plot_amper_files(
    txt_paths,
    output_path=None,
    f0_min=0,
    f0_max=300,
    title=None,
    labels=None,
    mode="segments_centered"
):
    """
    Main wrapper for AMPER plotting.

    Parameters
    ----------
    txt_paths : list of str
        Paths to AMPER .txt files.
    output_path : str or None
        Optional output image path.
    f0_min, f0_max : float
        Y-axis limits.
    title : str or None
        Plot title.
    labels : list of str or None
        Display labels for the plotted contours.
    mode : str
        Either:
        - "segments"          : x-axis in absolute time
        - "segments_centered" : x-axis as aligned syllable position
    """
    base_curves = []

    for p in txt_paths:
        if not os.path.exists(p):
            print(f"Missing AMPER file: {p}")
            continue
        base_curves.append(build_amper_curve(p))

    if not base_curves:
        print("No valid AMPER curves found.")
        return

    if mode == "segments":
        centered = False
    elif mode == "segments_centered":
        centered = True
    else:
        raise ValueError("mode must be 'segments' or 'segments_centered'.")

    plot_amper_segments_overlay(
        curves_raw=base_curves,
        labels=labels,
        output_path=output_path,
        f0_min=f0_min,
        f0_max=f0_max,
        title=title,
        centered=centered,
    )

    return base_curves


# ============================================================
# 3. DIRECTORY SCANNING AND FILE METADATA
# ============================================================

def parse_amper_filename_meta(path):
    """
    Extract metadata from an AMPER filename.

    Example
    -------
    06b6bwka1.txt

    Parsed as:
      - locality = '06b6'
      - context  = 'bwk'
      - ptype    = 'a'
      - rep      = 1
    """
    base = os.path.basename(path)
    stem, _ = os.path.splitext(base)

    if len(stem) < 7:
        raise ValueError(f"Filename too short for AMPER parsing: {base}")

    locality = stem[:4]
    rep_char = stem[-1]
    ptype_char = stem[-2].lower()
    context = stem[4:-2]

    if ptype_char not in ("a", "i"):
        ptype_char = "?"

    try:
        rep = int(rep_char)
    except ValueError:
        rep = None

    return locality, context, ptype_char, rep


def scan_amper_files(root_dir, only_ai=True):
    """
    Recursively scan root_dir and collect AMPER .txt files only from
    directories whose basename starts with 'wav_tgrid_txt_'.

    Returns
    -------
    pandas.DataFrame
        Columns:
        - path
        - locality
        - city
        - context
        - ptype
        - rep
    """
    rows = []

    for root, _, files in os.walk(root_dir):
        base_dir = os.path.basename(root)
        if not base_dir.startswith("wav_tgrid_txt_"):
            continue

        parent = os.path.basename(os.path.dirname(root))
        if "_" in parent:
            locality_from_dir, city = parent.split("_", 1)
        else:
            locality_from_dir = parent[:4]
            city = parent[4:]

        for name in files:
            if not name.lower().endswith(".txt"):
                continue

            full = os.path.join(root, name)

            try:
                locality_from_file, context, ptype, rep = parse_amper_filename_meta(full)
            except Exception:
                continue

            locality = locality_from_file or locality_from_dir

            if only_ai and ptype not in ("a", "i"):
                continue

            rows.append({
                "path": full,
                "locality": locality,
                "city": city,
                "context": context,
                "ptype": ptype,
                "rep": rep,
            })

    if not rows:
        print("No AMPER files found in 'wav_tgrid_txt_*' folders.")
        return pd.DataFrame(columns=["path", "locality", "city", "context", "ptype", "rep"])

    return pd.DataFrame(rows)


# ============================================================
# 4. BATCH PLOTTING: DECLARATIVE VS INTERROGATIVE EXAMPLES
# ============================================================

def batch_plot_amper_examples(
    root_dir,
    output_dir,
    max_plots_per_locality=5,
    f0_min=40,
    f0_max=200,
    city_filter=None,
    locality_filter=None
):
    """
    For each locality:
      - group files by (context, repetition)
      - keep only groups with both 'a' and 'i'
      - generate AMPER comparison plots
      - produce at most max_plots_per_locality plots

    Optional filters
    ----------------
    city_filter : list or None
        List of city names to include.
    locality_filter : list or None
        List of locality codes to include.
    """
    os.makedirs(output_dir, exist_ok=True)

    meta = scan_amper_files(root_dir, only_ai=True)
    if meta.empty:
        print("No valid AMPER files found for batch plotting.")
        return

    if city_filter is not None:
        meta = meta[meta["city"].isin(city_filter)]

    if locality_filter is not None:
        meta = meta[meta["locality"].isin(locality_filter)]

    if meta.empty:
        print("No files remaining after city/locality filtering.")
        return

    for loc, sub_loc in meta.groupby("locality"):
        plotted = 0
        city = sub_loc["city"].iloc[0] if "city" in sub_loc.columns else loc

        for (ctx, rep), sub_ctx in sub_loc.groupby(["context", "rep"]):
            sub_ai = sub_ctx[sub_ctx["ptype"].isin(["a", "i"])]
            if sub_ai["ptype"].nunique() < 2:
                continue

            paths = []
            for ptype in ["a", "i"]:
                cand = sub_ai[sub_ai["ptype"] == ptype]
                if cand.empty:
                    break
                paths.append(cand.iloc[0]["path"])
            else:
                safe_ctx = "".join(ch if ch.isalnum() else "_" for ch in ctx)
                out_name = f"{loc}_{safe_ctx}_r{rep}.png"
                out_path = os.path.join(output_dir, out_name)
                title = f"{city} - {ctx} (rep {rep})"

                plot_amper_files(
                    paths,
                    title=title,
                    output_path=out_path,
                    f0_min=f0_min,
                    f0_max=f0_max,
                    mode="segments_centered",
                )

                print(f"Plot saved: {out_path}")
                plotted += 1

                if plotted >= max_plots_per_locality:
                    break


# ============================================================
# 5. FEATURE EXTRACTION FOR STATISTICAL ANALYSIS
# ============================================================

def extract_features_from_amper_file(path):
    """
    Extract the two summary measures used in the statistical analyses.

    Measures
    --------
    1. mean_dur_first_half
       Mean syllable duration in the exact first half of the utterance,
       defined in temporal terms. If a syllable crosses the midpoint,
       only the portion falling within [0, T/2] is counted.

    2. mean_f0_attack
       Mean F0 over the first three syllables
       (up to 9 AMPER values: 3 points per syllable).

    Returns
    -------
    tuple
        (mean_dur_first_half, mean_f0_attack)
    """
    data = parse_amper_txt(path)

    dur_ms = np.asarray(data["dur_ms"], float)
    f0_points = np.asarray(data["f0_points"], float)

    if dur_ms.size == 0:
        return np.nan, np.nan

    total = float(dur_ms.sum())
    half = 0.5 * total

    end = np.cumsum(dur_ms)
    start = np.concatenate(([0.0], end[:-1]))

    overlap = np.clip(np.minimum(end, half) - start, 0, None)

    valid = overlap > 0
    if not np.any(valid):
        mean_dur_first_half = np.nan
    else:
        mean_dur_first_half = float(np.nanmean(overlap[valid]))

    n_syll = f0_points.shape[0]
    use_n = min(3, n_syll)

    if use_n == 0:
        mean_f0_attack = np.nan
    else:
        f0_first = f0_points[:use_n, :].reshape(-1)
        mean_f0_attack = float(np.nanmean(f0_first))

    return mean_dur_first_half, mean_f0_attack


# ============================================================
# 6. BATCH STATISTICS BY LOCALITY AND CONTEXT
# ============================================================

def batch_stats_amper(root_dir, alpha=0.05, min_per_type=2):
    """
    Run affirmative vs. interrogative comparisons for each
    (locality, context) combination.

    For each file, compute:
      - mean_dur_first_half
      - mean_f0_attack

    Then run Welch t-tests between 'a' and 'i'.

    Returns
    -------
    pandas.DataFrame
        Columns:
        locality, city, context,
        n_A, n_I,
        mean_dur_A, mean_dur_I, t_dur, p_dur, sig_dur,
        mean_f0_A, mean_f0_I, t_f0, p_f0, sig_f0
    """
    meta = scan_amper_files(root_dir, only_ai=True)
    if meta.empty:
        print("No valid AMPER files found for statistical analysis.")
        return pd.DataFrame()

    rows = []

    for (loc, ctx), sub in meta.groupby(["locality", "context"]):
        sub_ai = sub[sub["ptype"].isin(["a", "i"])]
        if sub_ai["ptype"].nunique() < 2:
            continue

        city = sub_ai["city"].iloc[0] if "city" in sub_ai.columns else loc

        dur_A, dur_I = [], []
        f0_A, f0_I = [], []

        for ptype, sub_t in sub_ai.groupby("ptype"):
            for _, row in sub_t.iterrows():
                mean_dur, mean_f0 = extract_features_from_amper_file(row["path"])
                if np.isnan(mean_dur) or np.isnan(mean_f0):
                    continue

                if ptype == "a":
                    dur_A.append(mean_dur)
                    f0_A.append(mean_f0)
                elif ptype == "i":
                    dur_I.append(mean_dur)
                    f0_I.append(mean_f0)

        if len(dur_A) < min_per_type or len(dur_I) < min_per_type:
            continue

        if len(dur_A) >= 2 and len(dur_I) >= 2:
            t_dur, p_dur = stats.ttest_ind(dur_A, dur_I, equal_var=False)
        else:
            t_dur, p_dur = np.nan, np.nan

        if len(f0_A) >= 2 and len(f0_I) >= 2:
            t_f0, p_f0 = stats.ttest_ind(f0_A, f0_I, equal_var=False)
        else:
            t_f0, p_f0 = np.nan, np.nan

        rows.append({
            "locality": loc,
            "city": city,
            "context": ctx,
            "n_A": len(dur_A),
            "n_I": len(dur_I),
            "mean_dur_A": np.mean(dur_A),
            "mean_dur_I": np.mean(dur_I),
            "t_dur": t_dur,
            "p_dur": p_dur,
            "sig_dur": bool(p_dur < alpha) if np.isfinite(p_dur) else False,
            "mean_f0_A": np.mean(f0_A) if f0_A else np.nan,
            "mean_f0_I": np.mean(f0_I) if f0_I else np.nan,
            "t_f0": t_f0,
            "p_f0": p_f0,
            "sig_f0": bool(p_f0 < alpha) if np.isfinite(p_f0) else False,
        })

    return pd.DataFrame(rows)


# ============================================================
# 7. BATCH STATISTICS AGGREGATED BY CONTEXT
# ============================================================

def batch_stats_amper_by_context(root_dir, alpha=0.05, min_per_type=2):
    """
    Run affirmative vs. interrogative comparisons aggregated by context,
    pooling across all localities.

    For each context:
      - collect all affirmative files
      - collect all interrogative files
      - compute:
          * mean_dur_first_half
          * mean_f0_attack
      - run Welch t-tests

    Returns
    -------
    pandas.DataFrame
        Columns:
        context,
        n_files_A, n_files_I,
        n_loc_A, n_loc_I,
        mean_dur_A, mean_dur_I, t_dur, p_dur, sig_dur,
        mean_f0_A, mean_f0_I, t_f0, p_f0, sig_f0
    """
    meta = scan_amper_files(root_dir, only_ai=True)
    if meta.empty:
        print("No valid AMPER files found for context-level analysis.")
        return pd.DataFrame()

    rows = []

    for ctx, sub in meta.groupby("context"):
        sub_ai = sub[sub["ptype"].isin(["a", "i"])]
        if sub_ai["ptype"].nunique() < 2:
            continue

        dur_A, dur_I = [], []
        f0_A, f0_I = [], []
        loc_A, loc_I = set(), set()

        for ptype, sub_t in sub_ai.groupby("ptype"):
            for _, row in sub_t.iterrows():
                mean_dur, mean_f0 = extract_features_from_amper_file(row["path"])
                if np.isnan(mean_dur) or np.isnan(mean_f0):
                    continue

                if ptype == "a":
                    dur_A.append(mean_dur)
                    f0_A.append(mean_f0)
                    loc_A.add(row["locality"])
                elif ptype == "i":
                    dur_I.append(mean_dur)
                    f0_I.append(mean_f0)
                    loc_I.add(row["locality"])

        if len(dur_A) < min_per_type or len(dur_I) < min_per_type:
            continue

        if len(dur_A) >= 2 and len(dur_I) >= 2:
            t_dur, p_dur = stats.ttest_ind(dur_A, dur_I, equal_var=False)
        else:
            t_dur, p_dur = np.nan, np.nan

        if len(f0_A) >= 2 and len(f0_I) >= 2:
            t_f0, p_f0 = stats.ttest_ind(f0_A, f0_I, equal_var=False)
        else:
            t_f0, p_f0 = np.nan, np.nan

        rows.append({
            "context": ctx,
            "n_files_A": len(dur_A),
            "n_files_I": len(dur_I),
            "n_loc_A": len(loc_A),
            "n_loc_I": len(loc_I),
            "mean_dur_A": np.mean(dur_A),
            "mean_dur_I": np.mean(dur_I),
            "t_dur": t_dur,
            "p_dur": p_dur,
            "sig_dur": bool(p_dur < alpha) if np.isfinite(p_dur) else False,
            "mean_f0_A": np.mean(f0_A) if f0_A else np.nan,
            "mean_f0_I": np.mean(f0_I) if f0_I else np.nan,
            "t_f0": t_f0,
            "p_f0": p_f0,
            "sig_f0": bool(p_f0 < alpha) if np.isfinite(p_f0) else False,
        })

    return pd.DataFrame(rows)


# ============================================================
# 8. EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":
    root_dir = "/Users/Federico/Desktop/Italia/Unzip"
    output_dir = "/Users/Federico/Desktop/Italia/plots_amper"

    # 1. Inspect file metadata
    meta_df = scan_amper_files(root_dir)
    print(meta_df.head())

    # 2. Plot declarative vs. interrogative examples for selected cities
    batch_plot_amper_examples(
        root_dir=root_dir,
        output_dir=output_dir,
        max_plots_per_locality=3,
        f0_min=40,
        f0_max=400,
        city_filter=["Battipaglia", "Pollina", "Policoro", "Prato"],
    )

    # 3. Statistics by locality and context
    stats_loc_ctx = batch_stats_amper(root_dir, alpha=0.05)
    stats_loc_ctx.to_csv(
        "/Users/Federico/Desktop/Italia/plots_amper/amper_stats_by_locality_context.csv",
        index=False
    )

    # 4. Statistics aggregated by context
    stats_ctx = batch_stats_amper_by_context(root_dir, alpha=0.05)
    stats_ctx.to_csv(
        "/Users/Federico/Desktop/Italia/plots_amper/amper_stats_by_context.csv",
        index=False
    )

In [ ]:
# ============================================================
# AMPER Intonation Visualizer
# ============================================================
#
# This script provides a lightweight visualization workflow for
# AMPER-style text files containing:
#
#     duration [ms]   energy [dB]   fo1 fo2 fo3 [Hz]
#     1   70   71   186 186 188
#     ...
#     values at:
#     t1 t2 t3 ... t_(3*N)
#
# Main features:
#   - robust parsing of AMPER .txt files
#   - reconstruction of absolute time in seconds
#   - segment-based plotting of vocalic triplets
#   - optional duration-weighted syllable alignment
#   - optional continuous interpolation with PCHIP
#   - colour encoding of syllabic rate
#   - line thickness encoding of energy
#
# Visual encoding:
#   - colour   = syllabic rate (faster = redder, slower = greener)
#   - thickness = syllabic energy (higher dB = thicker line)
#
# Plotting modes:
#   - "segments": classic AMPER-like segment display
#   - "smooth": continuous interpolated contour
#
# Requirements:
#   - numpy
#   - matplotlib
#   - scipy
#
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator


# ============================================================
# 1. AMPER PARSING
# ============================================================

def parse_amper_txt(path):
    """
    Parse an AMPER-style text file.

    Expected structure:
        duration [ms]   energy [dB]   fo1 fo2 fo3 [Hz]
        1   70   71   186 186 188
        ...
        values at:
        2616 4169 5721 ...

    Returns
    -------
    dict
        Dictionary with:
        - 'syll'      : syllable indices (1..N)
        - 'dur_ms'    : syllable durations in ms
        - 'energy'    : syllable energy values in dB
        - 'f0_points' : array of shape (N, 3)
        - 'times_raw' : array of length 3*N with AMPER timestamps
    """
    syll_idx = []
    dur_ms = []
    energy = []
    f0_points = []
    times_raw = []

    mode = "start"

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue

            if "duration [ms]" in s and "fo3" in s:
                mode = "rows"
                continue

            if s.lower().startswith("values at"):
                mode = "times"
                continue

            if mode == "rows":
                parts = s.split()
                if not parts or not parts[0].isdigit():
                    continue
                if len(parts) < 5:
                    continue

                idx = int(parts[0])
                dur = float(parts[1])

                try:
                    en = float(parts[2])
                except ValueError:
                    en = np.nan

                f0_vals = []
                for x in parts[-3:]:
                    try:
                        f0_vals.append(float(x))
                    except ValueError:
                        f0_vals.append(np.nan)

                syll_idx.append(idx)
                dur_ms.append(dur)
                energy.append(en)
                f0_points.append(f0_vals)

            elif mode == "times":
                for tok in s.split():
                    if tok.isdigit():
                        times_raw.append(int(tok))

    syll = np.array(
        syll_idx if syll_idx else list(range(1, len(dur_ms) + 1)),
        dtype=int
    )
    dur_ms = np.array(dur_ms, dtype=float)
    energy = np.array(energy, dtype=float)
    f0_points = np.asarray(f0_points, dtype=float)
    times_raw = np.array(times_raw, dtype=float)

    return {
        "syll": syll,
        "dur_ms": dur_ms,
        "energy": energy,
        "f0_points": f0_points,
        "times_raw": times_raw,
    }


# ============================================================
# 2. ABSOLUTE TIME BASE
# ============================================================

def build_amper_timebase(dur_ms, times_raw):
    """
    Build an absolute time axis in seconds from:
      - syllable durations in ms
      - raw AMPER timestamps

    Strategy
    --------
    1. Normalize raw AMPER times to [0, 1].
    2. Rescale them to [0, total_duration_sec], so that the
       total curve duration matches the sum of syllable durations.
    """
    dur_ms = np.asarray(dur_ms, float)
    times_raw = np.asarray(times_raw, float)

    total_sec = float(dur_ms.sum()) / 1000.0 if dur_ms.size else 1.0

    if times_raw.size == 0:
        n = max(1, 3 * len(dur_ms))
        return np.linspace(0.0, total_sec, n)

    t_min = np.nanmin(times_raw)
    t_max = np.nanmax(times_raw)

    if not np.isfinite(t_min) or not np.isfinite(t_max) or t_max <= t_min:
        n = max(1, 3 * len(dur_ms))
        return np.linspace(0.0, total_sec, n)

    rel = (times_raw - t_min) / (t_max - t_min)
    return rel * total_sec


# ============================================================
# 3. RAW CURVE CONSTRUCTION
# ============================================================

def build_amper_curve(path):
    """
    Convert an AMPER file into a raw F0 curve with 3*N points
    in absolute time (seconds).

    Returns
    -------
    dict
        - time_raw    : 3*N time values in seconds
        - f0_raw      : 3*N F0 values
        - speed_raw   : 3*N syllabic-rate values
        - energy_raw  : 3*N energy values
        - rate_syll   : N syllabic-rate values
        - syll_index  : syllable indices
        - dur_ms      : syllable durations
        - energy_syll : syllable energy values
        - source      : source file path
    """
    data = parse_amper_txt(path)
    dur_ms = data["dur_ms"]
    energy_syll = data["energy"]
    f0_points = data["f0_points"]
    times_raw = data["times_raw"]

    n_syll = len(dur_ms)
    if n_syll == 0:
        raise ValueError(f"No syllables found in: {path}")

    time_raw = build_amper_timebase(dur_ms, times_raw)

    if time_raw.size != 3 * n_syll:
        print(
            f"Warning: expected {3 * n_syll} timestamps, found {time_raw.size} "
            f"in {os.path.basename(path)}. Rescaling from syllable count."
        )
        time_raw = np.linspace(time_raw.min(), time_raw.max(), 3 * n_syll)

    if f0_points.shape[0] != n_syll or f0_points.shape[1] != 3:
        raise ValueError(
            f"Unexpected F0 structure in {path}: expected (N, 3), "
            f"found {f0_points.shape}"
        )

    f0_raw = f0_points.reshape(-1)

    dur_sec = dur_ms / 1000.0
    with np.errstate(divide="ignore", invalid="ignore"):
        rate_syll = np.where(dur_sec > 0, 1.0 / dur_sec, 0.0)

    syll_for_point = np.repeat(np.arange(n_syll), 3)
    speed_raw = rate_syll[syll_for_point]
    energy_raw = energy_syll[syll_for_point]

    return {
        "time_raw": time_raw,
        "f0_raw": f0_raw,
        "speed_raw": speed_raw,
        "energy_raw": energy_raw,
        "rate_syll": rate_syll,
        "syll_index": data["syll"],
        "dur_ms": dur_ms,
        "energy_syll": energy_syll,
        "source": path,
    }


# ============================================================
# 4. OPTIONAL CONTINUOUS SMOOTHING
# ============================================================

def smooth_amper_curve(curve, npoints=2000):
    """
    Build a continuous interpolated contour in absolute time.

    PCHIP interpolation is used for F0 in order to preserve shape
    and avoid overshoot. Syllabic rate and energy are linearly
    interpolated onto the same time grid.
    """
    t = np.asarray(curve["time_raw"], float)
    f0 = np.asarray(curve["f0_raw"], float)
    sp = np.asarray(curve["speed_raw"], float)
    en = np.asarray(curve["energy_raw"], float)

    order = np.argsort(t)
    t = t[order]
    f0 = f0[order]
    sp = sp[order]
    en = en[order]

    uniq_t, uniq_idx = np.unique(t, return_index=True)
    f0u = f0[uniq_idx]
    spu = sp[uniq_idx]
    enu = en[uniq_idx]

    t_smooth = np.linspace(uniq_t.min(), uniq_t.max(), int(max(npoints, 200)))

    try:
        if uniq_t.size >= 3:
            spline_f0 = PchipInterpolator(uniq_t, f0u)
            f0_smooth = spline_f0(t_smooth)
        else:
            f0_smooth = np.interp(t_smooth, uniq_t, f0u)
    except Exception:
        t_smooth = uniq_t
        f0_smooth = f0u

    speed_smooth = np.interp(t_smooth, uniq_t, spu)
    energy_smooth = np.interp(t_smooth, uniq_t, enu)

    return {
        "time": t_smooth,
        "f0": f0_smooth,
        "speed": speed_smooth,
        "energy": energy_smooth,
        "source": curve.get("source", ""),
    }


# ============================================================
# 5. LABELING HELPERS
# ============================================================

def infer_amper_label_from_name(path):
    """
    Extract a human-readable label from the file name.

    Rule:
      - take the file stem
      - if an underscore is present, keep the substring before
        the first underscore
      - capitalize the first letter

    Examples
    --------
    calvaruso_uno.txt -> Calvaruso
    quasimodo_due.txt -> Quasimodo
    """
    base = os.path.basename(path)
    stem, _ = os.path.splitext(base)

    if "_" in stem:
        label = stem.split("_", 1)[0]
    else:
        label = stem

    if not label:
        return "?"

    return label[0].upper() + label[1:]


def normalize_scalar(x, mn, mx):
    """
    Normalize a scalar to the [0, 1] interval.
    """
    if not np.isfinite(x) or not np.isfinite(mn) or not np.isfinite(mx) or mx <= mn:
        return 0.0
    return float((x - mn) / (mx - mn + 1e-12))


# ============================================================
# 6. SEGMENT-BASED PLOTTING
# ============================================================

def plot_amper_segments_overlay(
    curves_raw,
    labels=None,
    output_path=None,
    f0_min=0,
    f0_max=300,
    title=None,
    centered=False
):
    """
    Plot one or more AMPER contours using segment-based vocalic triplets.

    Parameters
    ----------
    curves_raw : list of dict
        Raw curves returned by build_amper_curve().
    labels : list of str or None
        Labels for the legend.
    output_path : str or None
        Path to save the figure.
    f0_min, f0_max : float or None
        Y-axis limits. If None, limits are inferred from the data.
    title : str or None
        Plot title.
    centered : bool
        If False, use absolute time on the x-axis.
        If True, align syllables by syllable position and scale segment
        width according to relative syllable duration.

    Notes
    -----
    Visual encoding:
      - colour   = syllabic rate (RdYlGn_r: faster = redder)
      - thickness = syllabic energy
    """
    if not curves_raw:
        print("No curves available for plotting.")
        return

    if labels is None:
        visible_labels = [infer_amper_label_from_name(c.get("source", "")) for c in curves_raw]
    else:
        visible_labels = list(labels)

    curve_stats = []
    for i, c in enumerate(curves_raw):
        f0_vals = np.asarray(c["f0_raw"], float)
        mean_f0 = float(np.nanmean(f0_vals)) if f0_vals.size else np.nan
        curve_stats.append((i, mean_f0))

    order = [i for i, _ in sorted(curve_stats, key=lambda x: x[1], reverse=True)]
    curves_ord = [curves_raw[i] for i in order]
    labels_ord = [visible_labels[i] for i in order]

    all_speed = np.concatenate([np.asarray(c["speed_raw"], float) for c in curves_ord])
    all_energy = np.concatenate([np.asarray(c["energy_raw"], float) for c in curves_ord])
    speed_min, speed_max = np.nanmin(all_speed), np.nanmax(all_speed)
    energy_min, energy_max = np.nanmin(all_energy), np.nanmax(all_energy)

    all_f0 = np.concatenate([np.asarray(c["f0_raw"], float) for c in curves_ord])
    if f0_min is None:
        f0_min = float(np.nanmin(all_f0)) - 5.0
    if f0_max is None:
        f0_max = float(np.nanmax(all_f0)) + 5.0

    max_n_syll = 0
    for c in curves_ord:
        n_syll = min(len(c["dur_ms"]), len(c["f0_raw"]) // 3)
        max_n_syll = max(max_n_syll, n_syll)

    fig, ax = plt.subplots(figsize=(9, 4))

    for curve, label in zip(curves_ord, labels_ord):
        t = np.asarray(curve["time_raw"], float)
        f0 = np.asarray(curve["f0_raw"], float)
        sp = np.asarray(curve["speed_raw"], float)
        en = np.asarray(curve["energy_raw"], float)
        dur = np.asarray(curve["dur_ms"], float)

        n_syll = min(len(dur), len(f0) // 3)

        if centered and n_syll > 0:
            mean_dur = np.nanmean(dur)
            if not np.isfinite(mean_dur) or mean_dur <= 0:
                mean_dur = 1.0
            dur_norm = np.clip(dur / mean_dur, 0.3, 1.7)
            base_width = 0.6
        else:
            dur_norm = None
            base_width = None

        for s_idx in range(n_syll):
            k0 = 3 * s_idx
            k1 = k0 + 1
            k2 = k0 + 2

            seg_f0 = f0[[k0, k1, k2]]

            if centered:
                center = s_idx + 1
                width = base_width * dur_norm[s_idx] if dur_norm is not None else 0.6
                half_w = width / 2.0
                seg_x = np.array([center - half_w, center, center + half_w])
            else:
                seg_x = t[[k0, k1, k2]]

            sp_val = sp[k0]
            en_val = en[k0]

            s_norm = normalize_scalar(sp_val, speed_min, speed_max)
            e_norm = normalize_scalar(en_val, energy_min, energy_max)
            lw = 2.0 + 6.0 * e_norm

            ax.plot(
                seg_x,
                seg_f0,
                color=plt.cm.RdYlGn_r(s_norm),
                linewidth=lw,
                alpha=0.85,
                solid_capstyle="round",
                solid_joinstyle="round",
                antialiased=True,
            )

        mean_f0 = float(np.nanmean(f0))
        mean_vowel_dur = float(np.nanmean(curve["dur_ms"])) if len(curve["dur_ms"]) else np.nan

        legend_label = (
            f"{label}\n"
            f"Mean f0: {mean_f0:.0f} Hz\n"
            f"Mean vowel dur: {mean_vowel_dur:.0f} ms"
        )

        ax.plot(
            [],
            [],
            color="black",
            linewidth=2.5,
            solid_capstyle="round",
            solid_joinstyle="round",
            label=legend_label,
        )

    ax.set_ylim(f0_min, f0_max)

    if centered:
        ax.set_xlim(0.5, max_n_syll + 1.0)
        ax.set_xlabel("Syllable position (duration-weighted)")
    else:
        ax.set_xlabel("Time (s)")

    ax.set_ylabel("F0 (Hz)")

    if title:
        ax.set_title(title)

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        title="Contour",
        frameon=False,
    )

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {os.path.basename(output_path)}")

    plt.show()


# ============================================================
# 7. CONTINUOUS PLOTTING
# ============================================================

def plot_amper_smooth_overlay(
    curves_smooth,
    labels=None,
    output_path=None,
    f0_min=0,
    f0_max=300,
    title=None
):
    """
    Plot one or more continuously interpolated AMPER contours.

    Parameters
    ----------
    curves_smooth : list of dict
        Smoothed curves returned by smooth_amper_curve().
    labels : list of str or None
        Labels for the legend.
    output_path : str or None
        Path to save the figure.
    f0_min, f0_max : float or None
        Y-axis limits. If None, limits are inferred from the data.
    title : str or None
        Plot title.
    """
    if not curves_smooth:
        print("No curves available for plotting.")
        return

    if labels is None:
        labels = [infer_amper_label_from_name(c.get("source", "")) for c in curves_smooth]

    all_speed = np.concatenate([np.asarray(c["speed"], float) for c in curves_smooth])
    all_energy = np.concatenate([np.asarray(c["energy"], float) for c in curves_smooth])
    speed_min, speed_max = np.nanmin(all_speed), np.nanmax(all_speed)
    energy_min, energy_max = np.nanmin(all_energy), np.nanmax(all_energy)

    all_f0 = np.concatenate([np.asarray(c["f0"], float) for c in curves_smooth])
    if f0_min is None:
        f0_min = float(np.nanmin(all_f0)) - 5.0
    if f0_max is None:
        f0_max = float(np.nanmax(all_f0)) + 5.0

    fig, ax = plt.subplots(figsize=(9, 4))

    for curve, label in zip(curves_smooth, labels):
        t = np.asarray(curve["time"], float)
        f0 = np.asarray(curve["f0"], float)
        sp = np.asarray(curve["speed"], float)
        en = np.asarray(curve["energy"], float)

        for i in range(len(t) - 1):
            seg_x = t[i:i + 2]
            seg_y = f0[i:i + 2]

            s_norm = normalize_scalar(sp[i], speed_min, speed_max)
            e_norm = normalize_scalar(en[i], energy_min, energy_max)
            lw = 1.5 + 5.0 * e_norm

            ax.plot(
                seg_x,
                seg_y,
                color=plt.cm.RdYlGn_r(s_norm),
                linewidth=lw,
                alpha=0.9,
                solid_capstyle="round",
                antialiased=True,
            )

        ax.plot([], [], color="black", linewidth=2.5, label=label)

    ax.set_ylim(f0_min, f0_max)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("F0 (Hz)")

    if title:
        ax.set_title(title)

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        title="Contour",
        frameon=False,
    )

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {os.path.basename(output_path)}")

    plt.show()


# ============================================================
# 8. MAIN WRAPPER
# ============================================================

def plot_amper_files(
    txt_paths,
    output_path=None,
    f0_min=0,
    f0_max=300,
    title=None,
    npoints=2000,
    labels=None,
    mode="segments",
    centered=False
):
    """
    Main wrapper for AMPER plotting.

    Parameters
    ----------
    txt_paths : list of str
        Input AMPER file paths.
    output_path : str or None
        Path to save the figure.
    f0_min, f0_max : float or None
        Y-axis limits. Use None for automatic bounds.
    title : str or None
        Plot title.
    npoints : int
        Number of interpolation points for smooth mode.
    labels : list of str or None
        Labels for the legend.
    mode : str
        Either:
        - "segments" : AMPER-like segment-based display
        - "smooth"   : continuous interpolated contour
    centered : bool
        Only used in segment mode:
        - False: absolute time on x-axis
        - True : duration-weighted syllable alignment
    """
    base_curves = []

    for p in txt_paths:
        if not os.path.exists(p):
            print(f"Missing AMPER file: {p}")
            continue
        base_curves.append(build_amper_curve(p))

    if not base_curves:
        print("No valid AMPER curves found.")
        return

    if mode == "smooth":
        curves = [smooth_amper_curve(c, npoints=npoints) for c in base_curves]
        plot_amper_smooth_overlay(
            curves_smooth=curves,
            labels=labels,
            output_path=output_path,
            f0_min=f0_min,
            f0_max=f0_max,
            title=title,
        )
        return curves

    if mode != "segments":
        raise ValueError("mode must be either 'segments' or 'smooth'.")

    plot_amper_segments_overlay(
        curves_raw=base_curves,
        labels=labels,
        output_path=output_path,
        f0_min=f0_min,
        f0_max=f0_max,
        title=title,
        centered=centered,
    )
    return base_curves


# ============================================================
# 9. EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":

    plot_amper_files(
        [
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/calvaruso_uno.txt",
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/piccini_uno.txt",
        ],
        title="Alle fronde dei salici - E come potevamo noi cantare",
        output_path="/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/alle_fronde_e_come_potevamo_noi_cantare_cp.png",
        f0_min=0,
        f0_max=300,
        mode="segments",
        centered=True,
    )

    plot_amper_files(
        [
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/ottonello_uno.txt",
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/quasimodo_uno.txt",
        ],
        title="Alle fronde dei salici - E come potevamo noi cantare",
        output_path="/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/alle_fronde_e_come_potevamo_noi_cantare_oq.png",
        f0_min=0,
        f0_max=300,
        mode="segments",
        centered=True,
    )

    plot_amper_files(
        [
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/calvaruso_due.txt",
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/piccini_due.txt",
        ],
        title="Alle fronde dei salici - crocifisso sul palo del telegrafo?",
        output_path="/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/alle_fronde_crocifisso_sul_palo_cp.png",
        f0_min=0,
        f0_max=300,
        mode="segments",
        centered=True,
    )

    plot_amper_files(
        [
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/ottonello_due.txt",
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/quasimodo_due.txt",
        ],
        title="Alle fronde dei salici - crocifisso sul palo del telegrafo?",
        output_path="/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/alle_fronde_crocifisso_sul_palo_oq.png",
        f0_min=0,
        f0_max=300,
        mode="segments",
        centered=True,
    )

    plot_amper_files(
        [
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/calvaruso_tre.txt",
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/piccini_tre.txt",
        ],
        title="Alle fronde dei salici - oscillavano lievi al triste vento",
        output_path="/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/alle_fronde_oscillavano_lievi_cp.png",
        f0_min=0,
        f0_max=300,
        mode="segments",
        centered=True,
    )

    plot_amper_files(
        [
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/ottonello_tre.txt",
            "/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/amperfox_amperdat/Alle_fronde/quasimodo_tre.txt",
        ],
        title="Alle fronde dei salici - oscillavano lievi al triste vento",
        output_path="/Users/Federico/Desktop/materiali_tesi/Convegni/FYP/alle_fronde_oscillavano_lievi_oq.png",
        f0_min=0,
        f0_max=300,
        mode="segments",
        centered=True,
    )